# Day 13: Pandas 基础 —— DataFrame、索引、筛选、读写

> **目标**: 掌握 Pandas 最核心的基础操作，理解 DataFrame 是「带索引的二维表格」。
> **前置**: Day 11 的 NumPy 基础（Pandas 底层是 NumPy 数组）
> **数据**: `../data/sales.csv`（500 行，10 列）

## 1. DataFrame 创建 —— 从 Python 结构到表格

Pandas 的 `DataFrame` 是数据岗最常用数据结构。它 = NumPy 数组 + 行索引(index) + 列名(columns)。

In [1]:
import pandas as pd
import numpy as np

# 从字典创建（最常用）
data = {
    "name": ["Alice", "Bob", "Charlie"],
    "age": [25, 30, 35],
    "score": [85.5, 92.0, 78.0]
}
df = pd.DataFrame(data)
print(df)
print(type(df))   # <class 'pandas.core.frame.DataFrame'>

# 从 list of dicts 创建（和 csv.DictReader 输出一致）
records = [
    {"name": "Alice", "age": 25},
    {"name": "Bob", "age": 30},
]
df2 = pd.DataFrame(records)
print(df2)

# 从 NumPy 数组创建
arr = np.array([[1, 2, 3], [4, 5, 6]])
df3 = pd.DataFrame(arr, columns=["A", "B", "C"], index=["x", "y"])
print(df3)

      name  age  score
0    Alice   25   85.5
1      Bob   30   92.0
2  Charlie   35   78.0
<class 'pandas.core.frame.DataFrame'>
    name  age
0  Alice   25
1    Bob   30
   A  B  C
x  1  2  3
y  4  5  6


## 2. 基本属性 —— 先了解你的数据

拿到一个 DataFrame，第一件事是「看」它。

In [2]:
# 读取真实数据
df = pd.read_csv("../data/sales.csv")

print(df.shape)       # (行数, 列数)
print(df.columns)     # 列名 Index
print(df.index)       # 行索引（默认 0~499）
print(df.dtypes)      # 每列类型（object=字符串, int64, float64）

# 快速查看数据
print(df.head(3))     # 前3行
print(df.tail(3))     # 后3行
print(df.info())      # 非空计数 + 类型 + 内存占用
print(df.describe())  # 数值列的统计（count/mean/std/min/25%/50%/75%/max）

(500, 9)
Index(['order_id', 'customer_id', 'product', 'category', 'quantity', 'price',
       'order_date', 'country', 'total'],
      dtype='object')
RangeIndex(start=0, stop=500, step=1)
order_id       object
customer_id    object
product        object
category       object
quantity        int64
price           int64
order_date     object
country        object
total           int64
dtype: object
  order_id customer_id   product   category  quantity  price  order_date  \
0    O1000        C007  Keyboard  Accessory         2   1299  2024-01-01   
1    O1001        C004  Keyboard  Accessory         1     99  2024-01-01   
2    O1002        C005    Laptop   Computer         4     99  2024-01-02   

   country  total  
0  Germany   2598  
1       US     99  
2       US    396  
    order_id customer_id product   category  quantity  price  order_date  \
497    O1497        C004   Mouse  Accessory         1    999  2024-12-29   
498    O1498        C001   Phone     Mobile         2    999  

## 3. 列访问与 Series

DataFrame 的每一列是 `Series`（带索引的一维数组）。访问列是最频繁的操作。

In [3]:
# 单列访问 → Series
totals = df["total"]
print(type(totals))   # <class 'pandas.core.series.Series'>
print(totals.head())

# 多列访问 → DataFrame（用列表套）
subset = df[["order_id", "customer_id", "total"]]
print(subset.head())

# Series 的核心属性
print(totals.values)    # 底层 NumPy 数组
print(totals.index)     # 索引
print(totals.dtype)     # 类型
print(totals.shape)     # (500,)

# Series 统计（和 NumPy 一样）
print(totals.mean())
print(totals.max())
print(totals.argmax())   # 最大值的位置索引（不是 order_id）

<class 'pandas.core.series.Series'>
0    2598
1      99
2     396
3     396
4     495
Name: total, dtype: int64
  order_id customer_id  total
0    O1000        C007   2598
1    O1001        C004     99
2    O1002        C005    396
3    O1003        C007    396
4    O1004        C003    495
[2598   99  396  396  495  198 6495 1198 1495  297 1797 1797  897 1299
 2997 3996 2396 1999 5997  897 2997 2995 2598 4995 1198 1797  297 6495
  495  598  396  198 4995 9995  599 6495   99 5196 3998 3998   99  598
  495 2997  599  598   99 1299  897 1495   99  598 5196 1999  299 5997
 6495  396 1198 1196 1198 6495  598 5997 2997 1797  897 2396 1495  198
 3998 1797 2997  999 6495 1196 1198 1299  599  598 3996   99  999 6495
 7996   99 5196  598  297 1299 1495 1998 7996  598  599 1196  897 3998
  599 6495 5196  598 1198 1797 5997 4995 6495 1999   99 6495 9995 5196
 2997   99 2997  297 1495 7996  198 3996 5196  297 7996   99 1797 1299
 3998 5997  598 2997 1495 7996 6495  198  396 3897 7996   99  396   9

## 4. loc vs iloc —— 两种索引方式

| 方式 | 语法 | 索引依据 | 切片规则 |
|------|------|----------|----------|
| `loc` | `df.loc[行标签, 列标签]` | 标签名 | 两端都包含（含头含尾） |
| `iloc` | `df.iloc[行位置, 列位置]` | 整数位置 | 左闭右开（含头不含尾） |

⚠️ **常见错误**: `df.loc[0:3]` 取 4 行（含3），`df.iloc[0:3]` 取 3 行（不含3）。

In [4]:
# loc —— 用列名和行标签
print(df.loc[0, "total"])          # 第0行 total 列的值
print(df.loc[0:2, ["order_id", "total"]])  # 第0~2行（含2），两列

# iloc —— 用整数位置（和 Python 切片一样）
print(df.iloc[0, 8])                 # 第0行第8列（total 是第8列）
print(df.iloc[0:3, [0, 8]])          # 第0~2行（不含3），第0列和第8列

# 用 loc 做条件筛选（最常用的技巧）
print(df.loc[df["total"] > 5000, ["order_id", "customer_id", "total"]])

# 用 at/iat 取单个值（更快）
print(df.at[0, "total"])    # 同 loc[0, "total"] 但更快
print(df.iat[0, 8])           # 同 iloc[0, 8] 但更快

2598
  order_id  total
0    O1000   2598
1    O1001     99
2    O1002    396
2598
  order_id  total
0    O1000   2598
1    O1001     99
2    O1002    396
    order_id customer_id  total
6      O1006        C005   6495
18     O1018        C008   5997
27     O1027        C002   6495
33     O1033        C002   9995
35     O1035        C005   6495
..       ...         ...    ...
463    O1463        C001   7996
470    O1470        C005   9995
472    O1472        C003   5997
483    O1483        C006   9995
494    O1494        C005   9995

[84 rows x 3 columns]
2598
2598


## 5. 布尔筛选 —— DataFrame 的 WHERE

布尔筛选是 Pandas 最核心的操作，和 SQL 的 WHERE 等价。

In [5]:
# 单条件筛选
big_orders = df[df["total"] > 5000]
print(f"大订单数量: {len(big_orders)}")

# 多条件组合（必须用 & 和 |，不能用 Python 的 and/or）
# 每个条件都要用括号包起来！
uk_big = df[(df["country"] == "UK") & (df["total"] > 2000)]
print(f"UK 大订单: {len(uk_big)}")

# 或条件
us_or_fr = df[(df["country"] == "US") | (df["country"] == "France")]
print(f"US 或 France 订单: {len(us_or_fr)}")

# 更优雅的写法：isin
target = df[df["country"].isin(["US", "France", "Germany"])]
print(f"目标国家订单: {len(target)}")

# between（范围筛选）
mid = df[df["total"].between(1000, 3000)]
print(f"1000~3000 订单: {len(mid)}")

大订单数量: 84
UK 大订单: 76
US 或 France 订单: 205
目标国家订单: 271
1000~3000 订单: 188


## 6. 赋值修改 —— 新增列、改值、删列

⚠️ **重要**: 赋值时要用 `df.loc[条件, "列"] = 值`，不要链式赋值 `df[条件]["列"] = 值`（会导致 SettingWithCopyWarning）。

In [6]:
# 新增列
df["unit_price"] = df["total"] / df["quantity"]
print(df[["order_id", "total", "quantity", "unit_price"]].head())

# 修改特定行的值（loc 条件赋值）
df.loc[df["country"] == "US", "country"] = "USA"
print(df["country"].unique())

# 删除列
df_drop = df.drop(columns=["unit_price"])  # 返回新 DataFrame，原 df 不变
# 或 df.drop("unit_price", axis=1, inplace=True)  # 原地修改，不推荐 inplace
print(df_drop.columns)

# 重置索引（筛选后索引不连续时）
df_uk = df[df["country"] == "UK"].reset_index(drop=True)
print(df_uk.index)  # 0, 1, 2, ...

  order_id  total  quantity  unit_price
0    O1000   2598         2      1299.0
1    O1001     99         1        99.0
2    O1002    396         4        99.0
3    O1003    396         4        99.0
4    O1004    495         5        99.0
['Germany' 'USA' 'France' 'UK' 'China']
Index(['order_id', 'customer_id', 'product', 'category', 'quantity', 'price',
       'order_date', 'country', 'total'],
      dtype='object')
RangeIndex(start=0, stop=197, step=1)


## 7. 排序与去重

排序和去重是数据清洗的常用步骤。

In [7]:
# 按单列排序
df_sorted = df.sort_values("total", ascending=False)  # 降序
print(df_sorted[["order_id", "total"]].head())

# 多列排序（先按 category，再按 total 降序）
df_sorted2 = df.sort_values(["category", "total"], ascending=[True, False])
print(df_sorted2[["category", "order_id", "total"]].head())

# 去重
unique_countries = df["country"].unique()
print(f"国家列表: {unique_countries}")

# 按行去重（子集列）
df_nodup = df.drop_duplicates(subset=["customer_id", "country"])
print(f"去重前: {len(df)}, 去重后: {len(df_nodup)}")

    order_id  total
355    O1355   9995
370    O1370   9995
342    O1342   9995
324    O1324   9995
319    O1319   9995
      category order_id  total
33   Accessory    O1033   9995
238  Accessory    O1238   9995
324  Accessory    O1324   9995
342  Accessory    O1342   9995
432  Accessory    O1432   9995
国家列表: ['Germany' 'USA' 'France' 'UK' 'China']
去重前: 500, 去重后: 40


## 8. 类型转换 —— CSV 读进来全是字符串？

Pandas 读 CSV 时会自动推断类型，但日期、价格等经常需要手动转换。

In [8]:
# 检查当前类型
print(df.dtypes)

# 把字符串列转为数值（无法转换的变 NaN）
df["price_num"] = pd.to_numeric(df["price"], errors="coerce")
# errors="coerce": 转换失败 → NaN，不会报错

# 把日期字符串转为 datetime
df["date"] = pd.to_datetime(df["order_date"], errors="coerce")
print(df["date"].dtype)  # datetime64[ns]

# 从 datetime 提取年月
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
print(df[["order_date", "date", "year", "month"]].head())

# 类型转换回字符串（如导出前）
df["total_str"] = df["total"].astype(str)

order_id        object
customer_id     object
product         object
category        object
quantity         int64
price            int64
order_date      object
country         object
total            int64
unit_price     float64
dtype: object
datetime64[ns]
   order_date       date  year  month
0  2024-01-01 2024-01-01  2024      1
1  2024-01-01 2024-01-01  2024      1
2  2024-01-02 2024-01-02  2024      1
3  2024-01-03 2024-01-03  2024      1
4  2024-01-03 2024-01-03  2024      1


## 9. 读写 CSV —— 数据岗日常

```python
# 读 CSV（常用参数）
df = pd.read_csv("../data/sales.csv")
df = pd.read_csv("data.csv", encoding="utf-8", parse_dates=["order_date"])

# 写 CSV
df.to_csv("output.csv", index=False, encoding="utf-8")
# index=False 不写入行索引列（最常用）
```

**Pandas vs Python csv 模块**:
- 读 CSV 用 Pandas（自动分类型、支持大数据、方便分析）
- 流式处理/逐行清洗用 Python csv 模块（内存可控）
- 写 JSON 用 Python json 模块（Pandas 的 to_json 也可以，但控制粒度不如原生）

## 今日要点总结

| 操作 | 代码 | 注意 |
|------|------|------|
| 读取 CSV | `pd.read_csv("xxx.csv")` | 自动推断类型，日期需手动转 |
| 查看属性 | `df.shape / .columns / .dtypes / .info()` | info() 看非空计数 |
| 看数据 | `df.head() / .tail() / .describe()` | describe() 只看数值列 |
| 取列 | `df["col"]` → Series | 单列 |
| 取多列 | `df[["a","b"]]` → DataFrame | 双层列表 |
| loc | `df.loc[行标签, 列标签]` | 含头含尾，用标签 |
| iloc | `df.iloc[行位置, 列位置]` | 左闭右开，用整数 |
| 布尔筛选 | `df[df["x"] > 100]` | 多条件用 `&` + 括号 |
| isin | `df[df["x"].isin([...])]` | 多值匹配，比 `|` 优雅 |
| 赋值 | `df.loc[条件, "col"] = val` | 不用链式赋值 |
| 新增列 | `df["new"] = df["a"] / df["b"]` | 广播运算，不用循环 |
| 排序 | `df.sort_values("col", ascending=False)` | 多列传列表 |
| 去重 | `df.drop_duplicates(subset=[...])` | 按指定列去重 |
| 类型转换 | `pd.to_numeric(..., errors="coerce")` | 失败变 NaN 不崩 |
| 日期转换 | `pd.to_datetime(...)` | 提取年月用 `.dt.year` |

**核心心法**: Pandas 的每一列是 Series（带索引的 NumPy 数组），DataFrame 是 Series 的集合。看到循环操作 DataFrame，先想「能不能向量化」。